# 📱 Cara Mengubah Aplikasi Kivy Menjadi APK Android (Google Colab)

Notebook ini digunakan untuk membungkus kode Kivy kalkulator (`lesson2.py`) menjadi file **APK** yang bisa diinstall di HP Android.

### ⏱️ Catatan Waktu:
- **Langkah 1 (Install Dependencies)**: ~2-3 menit
- **Langkah 4 (Build APK)**: ~10-15 menit (karena download Android SDK & NDK pertama kali)

Jalankan sel secara berurutan dari langkah 1 sampai 5!

### 1️⃣ Install Tools & Library Sistem Linux

In [ ]:
# Install Buildozer, Cython, dan dependensi compiler Android
!pip install --upgrade buildozer cython virtualenv
!sudo apt-get update -qq
!sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev
print("\n✅ Instalasi dependensi selesai!")

### 2️⃣ Tulis Kode Kalkulator (`main.py`)
Buildozer mewajibkan file utama bernama `main.py`.

In [ ]:
%%writefile main.py
# =====================================================================
# LESSON 2 — Kalkulator Kivy
# =====================================================================
from kivy.app import App
from kivy.uix.button import Button
from kivy.uix.label import Label
from kivy.uix.screenmanager import ScreenManager, Screen
from kivy.uix.gridlayout import GridLayout
from kivy.uix.boxlayout import BoxLayout


class TombolAngka(Button):
    def __init__(self, kalkulator, **kwargs):
        super().__init__(**kwargs)
        self.kalkulator = kalkulator
        self.font_size = 28

    def on_press(self):
        self.kalkulator.tambah(self.text)


class TombolAksi(Button):
    def __init__(self, kalkulator, **kwargs):
        super().__init__(**kwargs)
        self.kalkulator = kalkulator
        self.font_size = 28

    def on_press(self):
        if self.text == 'C':
            self.kalkulator.hapus()
        elif self.text == '=':
            self.kalkulator.hitung()


class MainScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.layar = Label(text='0', font_size=40, size_hint=(1, 0.3))

        grid = GridLayout(cols=4, spacing=6)
        tombol_tombol = [
            '7', '8', '9', '+',
            '4', '5', '6', '-',
            '1', '2', '3', 'x',
            'C', '0', '=', ':',
        ]

        for text in tombol_tombol:
            if text in ['C', '=']:
                grid.add_widget(TombolAksi(kalkulator=self, text=text))
            else:
                grid.add_widget(TombolAngka(kalkulator=self, text=text))

        layout = BoxLayout(orientation='vertical', padding=8, spacing=8)
        layout.add_widget(self.layar)
        layout.add_widget(grid)

        self.add_widget(layout)

    def tambah(self, karakter):
        if self.layar.text == '0' or self.layar.text == 'Error':
            self.layar.text = karakter
        else:
            self.layar.text += karakter

    def hapus(self):
        self.layar.text = '0'

    def hitung(self):
        rumus = self.layar.text
        rumus_python = rumus.replace('x', '*').replace(':', '/')
        try:
            hasil = eval(rumus_python)
            self.layar.text = str(hasil)
        except:
            self.layar.text = 'Error'


class CalculatorApp(App):
    def build(self):
        sm = ScreenManager()
        sm.add_widget(MainScreen(name='main'))
        return sm


if __name__ == '__main__':
    CalculatorApp().run()


### 3️⃣ Buat & Sesuaikan Konfigurasi `buildozer.spec`

In [ ]:
import os, re

# Inisialisasi buildozer.spec bila belum ada
if not os.path.exists("buildozer.spec"):
    !buildozer init

with open("buildozer.spec", "r") as f:
    spec = f.read()

# Konfigurasi nama aplikasi & paket
spec = re.sub(r"^title = .*", "title = Kalkulator", spec, flags=re.MULTILINE)
spec = re.sub(r"^package\.name = .*", "package.name = kalkulatorkivy", spec, flags=re.MULTILINE)
spec = re.sub(r"^package\.domain = .*", "package.domain = org.algonova", spec, flags=re.MULTILINE)

# Requirements python & kivy
spec = re.sub(r"^requirements = .*", "requirements = python3,kivy", spec, flags=re.MULTILINE)

# Arsitektur arm64-v8a (standar HP masa kini, lebih cepat compile-nya)
spec = re.sub(r"^#?android\.archs = .*", "android.archs = arm64-v8a", spec, flags=re.MULTILINE)

# Otomatis menyetujui lisensi SDK Android
spec = re.sub(r"^#?android\.accept_sdk_license = .*", "android.accept_sdk_license = True", spec, flags=re.MULTILINE)

with open("buildozer.spec", "w") as f:
    f.write(spec)

print("✅ buildozer.spec berhasil disiapkan!")

### 4️⃣ Mulai Proses Build APK Android
Tunggu sampai selesai (akan muncul notifikasi `BUILD SUCCESSFUL`).

In [ ]:
# Menjalankan buildozer
!buildozer -v android debug

### 5️⃣ Download File APK ke Komputer / HP

In [ ]:
import glob
from google.colab import files

apk_list = glob.glob("bin/*.apk")
if apk_list:
    target_apk = apk_list[0]
    print(f"📦 Menemukan APK: {target_apk}")
    print("Mengunduh file...")
    files.download(target_apk)
else:
    print("❌ File APK tidak ditemukan di folder bin/. Periksa pesan error di langkah 4.")